# Imports

In [2]:
import pandas as pd
import json

In [3]:
import config
import os
os.environ["OPENAI_API_KEY"] = config.OPENAI_API_KEY
os.environ["OPENAI_API_BASE"] = config.OPENAI_API_BASE

In [8]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

In [4]:
from diffusers import DiffusionPipeline, AutoPipelineForText2Image
from diffusers.utils import load_image, make_image_grid

import torch
import torchvision

# Load Data

In [6]:
df = pd.read_csv('dummy_housing_data_v2.csv', index_col=0)
df.head(2)

,id,neighborhood_name,price,bedrooms,bathrooms,house_size,year_built,description,neighborhood_description
0,2,Willow Creek,650000,4,3.5,3000,2005,Discover the perfect family home in the desira...,Willow Creek is known for its top-rated school...
1,3,Riverfront Estates,950000,5,4.0,5000,2010,Luxury living awaits in the prestigious Riverf...,"Riverfront Estates offers waterfront views, pr..."


In [9]:
instructions = """
You are an architect that creates the exterior design of a house based on given data.
Provide a short exterior house description based on the given house information.
The exterior description should include the color of the walls, the roof and how the garden looks.

For example:
The 4-bedroom 3000sqft house evaluated at $500,000 is showing off its red brick walls with many trees as a backdrop. 
The house has a chimney and big windows.
"""

example_template = """
                Price: ${price}
                Bedrooms: {bedrooms}
                Bathrooms: {bathrooms}
                House Size: {house_size} sqft
                Year Built: {year_built}
                Description: {description}
                """

house_prompt = PromptTemplate(
    prefix = instructions,
    template=example_template,
    input_variables=[
         "price", "bedrooms", "bathrooms", "house_size",
         "year_built", "description"
    ]
)


In [10]:
model_name = "gpt-3.5-turbo"
temperature = 1.0
llm = ChatOpenAI(model_name=model_name, temperature=temperature, max_completion_tokens=50)

In [11]:
ext_desc = {}
for i, row in df.iterrows():
    
    response = llm.invoke(house_prompt.format(price=row['price'], bedrooms=row['bedrooms'],
                                                bathrooms=row['bathrooms'], house_size=row['house_size'],
                                                year_built=row['year_built'], description=row['description']))
    ext_desc[i] = response.content.strip()
print(ext_desc)

{0: 'The master suite boasts a walk-in closet and a luxurious en-suite bathroom with a soaking tub and separate shower. The three additional bedrooms are perfect for children or guests, and the finished basement provides extra living space. The three-car garage offers plenty', 1: 'The spacious backyard features a covered patio, perfect for outdoor entertaining. The home also boasts a media room, office, and three-car garage. Located near top-rated schools, shopping, and dining, this home offers the ultimate in luxury living. Don', 2: 'This well-maintained home is perfect for a growing family or anyone looking for a comfortable living space in a quiet community. The neighborhood offers a community pool, basketball court, and playground for residents to enjoy. Conveniently located near shopping centers, restaurants', 3: 'Amenities include hardwood floors, a gas fireplace, a bonus room that can be used as an office or playroom, and a 3-car garage. The beautifully landscaped backyard is pe

# Create Q&A Image Prompts

In [12]:
with open('qa_data.json', 'r') as file:
     qa_data = json.load(file)
print(qa_data)

{'questions': ['How many bed- and bathrooms do you want?', 'How many square feet?', 'How old can the house be?', 'What are the 3 most important things in the house?', 'Are there any amenities you want?', 'What kind of neighborhood do you want to live in?', 'How urban do you want your neighborhood to be?'], 'customer1': ['I want a three-bedroom, two-bathroom house.', 'Between 2000 to 3000 sqft.', 'I think it should be built in 1985 or later.', 'A spacious kitchen, a cozy living room, big windows.', 'A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.', 'A quiet neighborhood with good local schools, and convenient shopping options and easy access to major highways.', 'A balance between suburban tranquility and access to urban amenities like restaurants and theaters.'], 'customer2': ['A luxurious 5-bedroom, 3-bathroom house.', '5000 sqft or more.', 'Relatively new, built 2000 or later.', 'A gourmet kitchen, a winter garden, high ceilings.', 'A big ga

In [13]:
model_name = "gpt-3.5-turbo"
temperature = 0.3
llm = ChatOpenAI(model_name=model_name, temperature=temperature)

query_prompt = PromptTemplate(
    input_variables=["user_input"],
    template="""
    Extract a house description from the user chat history following below. 
    Make sure to include amenities, year built preferences and house size (bedrooms, bathrooms, sqft) 
    in the description. Keep it simple, maximum 2 sentences.

    ************User chat history:**************
     "{user_input}"
    """
)

def create_qa(questions, answers):
    qa = ""
    for i in range(len(questions)):
        qa += "Q: " + questions[i] + "\nA: " + answers[i] + "\n"

    return qa

qa_prompts = {}
for i, key in enumerate(['customer1', 'customer2', 'customer3']):
    user_input = create_qa(qa_data['questions'], qa_data[key])
    print(user_input)
    query = llm.invoke(query_prompt.format(user_input=user_input)).content
    print(query)
    print("")
    qa_prompts[i+1] = query

Q: How many bed- and bathrooms do you want?
A: I want a three-bedroom, two-bathroom house.
Q: How many square feet?
A: Between 2000 to 3000 sqft.
Q: How old can the house be?
A: I think it should be built in 1985 or later.
Q: What are the 3 most important things in the house?
A: A spacious kitchen, a cozy living room, big windows.
Q: Are there any amenities you want?
A: A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.
Q: What kind of neighborhood do you want to live in?
A: A quiet neighborhood with good local schools, and convenient shopping options and easy access to major highways.
Q: How urban do you want your neighborhood to be?
A: A balance between suburban tranquility and access to urban amenities like restaurants and theaters.

I am looking for a three-bedroom, two-bathroom house with a spacious kitchen, cozy living room, and big windows. Ideally, the house should be between 2000 to 3000 sqft in size, built in 1985 or later, and include 

# Create Images

included for reference (images where created in a cloud environment)

In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(
    "playgroundai/playground-v2-1024px-aesthetic",
    torch_dtype=torch.float16,
    use_safetensors=True,
    add_watermarker=False,
    variant="fp16"
)

In [ ]:
rand_gen = torch.manual_seed(42312981)
#rand_gen = torch.manual_seed(42312981)

for key, prompt in ext_desc.items():

    image  = pipe(prompt=prompt, guidance_scale=3.0, generator=rand_gen).images[0]
    image.save(f"./images/image_{key}.png")

In [ ]:
rand_gen = torch.manual_seed(42312981)
#rand_gen = torch.manual_seed(42312981)

for key, prompt in qa_prompts.items():

    image  = pipe(prompt=prompt, guidance_scale=3.0, generator=rand_gen).images[0]
    image.save(f"./images_qa/image_{key}.png")